In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import multiprocessing
import os
import pickle

try:
    cores = int(os.environ["SLURM_JOB_CPUS_PER_NODE"])
except:
    cores = multiprocessing.cpu_count()


os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(cores)

import jax
import jax.numpy as jnp
from astropy.table import Table
import numpy as np
import pandas as pd
import gpjax as gpx
from gpjax.kernels import RBF, Linear, Periodic
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.distributions import Normal
from tqdm import tqdm

from gallifrey.kernels import OrnsteinUhlenbeck
from gallifrey.kernelsearch import (
    KernelSearch,
    print_kernel_summary,
    get_trainables,
)
from gallifrey.mcmc import nuts_warmup, run_mcmc
from gallifrey.inference import log_likelihood_function

In [3]:
jax.config.update("jax_enable_x64", True)
rng_key = jax.random.PRNGKey(42)

## Load Data

In [4]:
df = (
    Table.read("../data/external/HATS_46b.fit")
    .to_pandas()
    .drop(columns=["FWB20", "e_FWB20"])  # not used in paper
)

t = df["Time"].to_numpy()
t_min = np.amin(t)
t -= t_min

# white lightcurve
white_lc = df["FWL"].to_numpy().T
white_lc_err = df["e_FWL"].to_numpy().T

# spectroscopic light curves
y = df.iloc[:, 3::2].to_numpy().T
yerr = df.iloc[:, 4::2].to_numpy().T

# mask out transit
mask = np.ones_like(t, dtype=bool)
mask[7:41] = False

# reference parameter and limb darkening parameter
reference = pd.read_csv("../data/external/HATS_46b_reference.csv")

## Get GP models for all light curves

In [5]:
kernel_library = [
    Linear(),
    RBF(),
    OrnsteinUhlenbeck(),
    Periodic(),
]

In [6]:
tree = KernelSearch(
    kernel_library,
    X=jnp.array(t[mask]),
    y=jnp.array(y[:, mask]),
    obs_stddev=jnp.amax(y[:, mask], axis=1),
    verbosity=1,
    criterion="bic",
)

model = tree.search(
    depth=7,
    n_leafs=3,
    patience=1,
)

Fitting Layer 1: 100%|██████████| 4/4 [01:04<00:00, 16.10s/it]


Layer 1 || Current top AICs: [231.02630531200666, 231.0263053678227, 233.02630614336613, 246.57201414677164]


Fitting Layer 2: 100%|██████████| 32/32 [12:00<00:00, 22.53s/it]


Layer 2 || Current top AICs: [233.02630538302392, 233.02630543866732, 233.02630580472973, 233.02630677026445, 233.02630682594247]


Fitting Layer 3:  48%|████▊     | 25/52 [09:50<10:37, 23.63s/it]


KeyboardInterrupt: 

In [ ]:
model_name = "hats46b_gpmodel"
with open(f"../data/processed/observational_data/gp_models/{model_name}", "wb") as file:
    pickle.dump(model, file)
# with open(f"../data/processed/observational_data/gp_models/{model_name}", "rb") as file:
#     model = pickle.load(file)
print_kernel_summary(model)

Kernel Summary

Kernel Structure: RBF

--------------------------------------------------------------------------------
Kernel               Property             Value                Trainable 
--------------------------------------------------------------------------------
RBF                 lengthscale          1479.0875            True      

                    variance             0.9831623            True      
--------------------------------------------------------------------------------


## Fit white curve parameter

In [8]:
def white_lc_model(t, params):
    central = orbits.keplerian.Central(
        mass=params[0],
        radius=params[1],
    )

    # The light curve calculation requires an orbit
    orbit = orbits.keplerian.Body(
        central=central,
        period=4.7423749,
        radius=params[4] * central.radius,
        inclination=jnp.deg2rad(params[2]),
        time_transit=params[3],
    )

    lc = LimbDarkLightCurve([params[5], 0.1171]).light_curve(orbit, t=t)
    return lc


white_lc_log_likelihood = log_likelihood_function(
    model,
    white_lc_model,
    t,
    white_lc,
    mask,
    fix_gp=False,
    compile=True,
    negative=True,
)

white_lc_solve = ScipyMinimize(fun=white_lc_log_likelihood, method="l-bfgs-b").run(
    {
        "gp_parameter": get_trainables(model, unconstrain=True),
        "lc_parameter": jnp.array([0.869, 0.894, 86.97, 0.075, 0.09773, 0.547]),
    }
)
white_lc_parameter = white_lc_solve.params["lc_parameter"]

## Define LC model

In [9]:
def get_lc_model(u2):
    def lc_model(t, params):
        central = orbits.keplerian.Central(
            mass=white_lc_parameter[0],
            radius=white_lc_parameter[1],
        )

        # The light curve calculation requires an orbit
        orbit = orbits.keplerian.Body(
            central=central,
            period=3.9501907,
            radius=params[0] * central.radius,
            inclination=jnp.deg2rad(white_lc_parameter[2]),
            time_transit=white_lc_parameter[3],
        )

        lc = LimbDarkLightCurve([params[1], u2]).light_curve(orbit, t=t)
        return lc

    return lc_model

## Define likelihood, prior, posterior

In [10]:
def get_logprob(gp_model, y, yerr, u1, u2, initial_position=None):
    if initial_position is None:
        initial_position = {
            "gp_parameter": get_trainables(gp_model, unconstrain=True),
            "lc_parameter": jnp.array([0.09, u1]),
        }

    param_priors = {
        "gp_parameter": Normal(
            loc=initial_position["gp_parameter"],
            scale=0.1 * jnp.abs(initial_position["gp_parameter"]),
        ),
        "lc_parameter": Normal(
            loc=initial_position["lc_parameter"],
            scale=[0.2, 0.05],
        ),
    }

    # define light curve model
    lc_model = jit(get_lc_model(u2))

    # update observational uncertainty data uncertainty
    gp_model = (
        gp_model.likelihood.replace(obs_stddev=jnp.amax(yerr[mask])) * model.prior
    )

    log_likelihood = log_likelihood_function(
        gp_model,
        lc_model,
        t,
        y,
        mask,
        fix_gp=False,
        compile=True,
    )

    @jit
    def log_priors(params):
        gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
        lc_log_priors = param_priors["lc_parameter"].log_prob(params["lc_parameter"])
        return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)

    @jit
    def log_probability(params):
        return log_likelihood(params) + log_priors(params)

    return log_probability, initial_position

## Fits

In [11]:
parameter_solutions = []
for i in tqdm(range(len(y))):
    log_probability, initial_position = get_logprob(
        model,
        y[i],
        yerr[i],
        reference["u1"].iloc[i],
        reference["u2"].iloc[i],
    )

    solve = ScipyMinimize(
        fun=jit(lambda par: -log_probability(par)), method="l-bfgs-b"
    ).run(initial_position)
    parameter_solutions.append(solve.params)

 52%|█████▏    | 13/25 [00:37<00:35,  2.98s/it]

In [ ]:
import matplotlib.pyplot as plt
from gallifrey.kernelsearch import set_trainables
from gallifrey.inference import calculate_predictive_dist

i = -3
gp_model = set_trainables(model, parameter_solutions[i]["gp_parameter"])
lc_model = get_lc_model(reference["u2"].iloc[i])(
    t, parameter_solutions[i]["lc_parameter"]
)

gp_dist = calculate_predictive_dist(
    gp_model,
    t,
    x=t[mask],
    y=y[i][mask],
)

predictive_mean = gp_dist.mean()
predictive_std = gp_dist.stddev()


plt.scatter(t, y[i])
plt.plot(t, predictive_mean)
plt.plot(t, lc_model + predictive_mean)
plt.fill_between(
    t,
    predictive_mean + lc_model + predictive_std,
    predictive_mean + lc_model - predictive_std,
    alpha=0.5,
)

In [ ]:
import matplotlib.pyplot as plt

Rps = np.array([sol["lc_parameter"][0] for sol in parameter_solutions])
plt.scatter(range(len(y)), Rps - np.mean(Rps))
plt.scatter(range(len(y)), reference["Rp"] - np.mean(reference["Rp"]))

plt.figure()
plt.scatter(
    range(len(y)),
    ((reference["Rp"] - np.mean(reference["Rp"])) - (Rps - np.mean(Rps)))
    / reference["Rp"],
)

## Run MCMC

In [ ]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 300
num_samples = 80
num_chains = cores

In [ ]:
rng_key, warmup_key = jax.random.split(rng_key, 2)

# run nuts adaption on white lc
log_probability, initial_position = get_logprob(
    model,
    white_lc,
    white_lc_err,
    0.5326,
    0.0862,
)

state, parameters = nuts_warmup(
    warmup_key,
    white_lc_log_likelihood,
    initial_position,
    num_steps=num_adapt,
)

In [ ]:
chains = {"gp_parameter": [], "lc_parameter": []}

for i in tqdm(range(len(y))):
    log_probability, initial_position = get_logprob(
        model,
        y[i],
        yerr[i],
        reference["u1_linear"].iloc[i],
        reference["u2"].iloc[i],
        initial_position=parameter_solutions[i],
    )

    # define initial positions and add scatter
    initial_positions = {}
    for key, value in initial_position.items():
        rng_key, scatter_key = jax.random.split(rng_key)
        initial_positions[key] = jnp.tile(
            value, (num_chains, 1)
        ) + 0.05 * value * jax.random.normal(
            scatter_key, shape=(num_chains, value.size)
        )

    rng_key, sample_key = jax.random.split(rng_key, 2)

    final_state, state_history, info_history = run_mcmc(
        sample_key,
        log_probability,
        parameters,
        initial_positions,
        num_steps=num_samples,
    )

    for par in ["gp_parameter", "lc_parameter"]:
        chain = np.array(state_history.position[par])
        chains[par].append(chain)

np.savez(f"saved/{model_name}_parameter.npz", **chains)